In [2]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, ParameterSampler

import joblib

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False


In [3]:

LABEL = "fraud"

TRAIN_PATH = "../../5DATA/dataset/TRAIN_STAGE2"
TEST_PATH  = "../../5DATA/dataset/TEST_STAGE2"

OUT_DIR = "artifacts/stage2_models"
OUT_DIR_METRICS = "artifacts/stage2_metrics"


In [4]:

from pathlib import Path

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUT_DIR_METRICS).mkdir(parents=True, exist_ok=True)

In [5]:

def load_stage_df(path: str, label: str = LABEL):
    df = pd.read_parquet(path)
    if label not in df.columns:
        raise KeyError(f"Missing label column: {label}")
    X = df.drop(columns=[label])
    y = df[label].astype(np.int8).to_numpy()
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X, y


In [6]:
def topk_metrics(y_true, score, top_pct_list=(0.001, 0.002, 0.005, 0.01)):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)
    n = len(y)
    base_rate = float(y.mean()) if n else np.nan
    order = np.argsort(-s)
    y_sorted = y[order]

    rows = []
    for p in top_pct_list:
        k = max(int(np.ceil(n * p)), 1)
        top_y = y_sorted[:k]
        prec = float(top_y.mean())
        rec = float(top_y.sum() / max(y.sum(), 1))
        lift = float(prec / base_rate) if base_rate and base_rate > 0 else np.nan
        rows.append({"top_pct": p, "k": k, "precision": prec, "recall": rec, "lift": lift, "base_rate": base_rate})
    return pd.DataFrame(rows)


def evaluate_metrics(y_true, score):
    return {
        "auc": float(roc_auc_score(y_true, score)),
        "prauc": float(average_precision_score(y_true, score)),
        "base_rate": float(np.mean(y_true)),
    }


In [7]:

def fit_logit(X_tr, y_tr):
    model = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced",
        ))
    ])
    model.fit(X_tr, y_tr)
    return model


def fit_hgb(X_tr, y_tr):
    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=200,
        l2_regularization=0.0,
        random_state=42,
    )
    model.fit(X_tr, y_tr)
    return model


def fit_lgb_small(X_tr, y_tr):
    if not HAS_LGB:
        raise RuntimeError("lightgbm is not available in this environment.")
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=6,
        min_data_in_leaf=300,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr)
    return model


def predict_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return 1 / (1 + np.exp(-s))
    return model.predict(X)


In [8]:
X_tr, y_tr = load_stage_df(TRAIN_PATH)
X_te, y_te = load_stage_df(TEST_PATH)

print("train:", X_tr.shape, "test:", X_te.shape)
print("base_rate train:", float(y_tr.mean()), "test:", float(y_te.mean()))


train: (608430, 64) test: (113943, 64)
base_rate train: 0.010794996959387276 test: 0.01832495194965904


In [9]:

from sklearn.metrics import classification_report

candidates = [
    ("logit", fit_logit),
    ("hgb", fit_hgb),
]
if HAS_LGB:
    candidates.append(("lgb_small", fit_lgb_small))

results = []
topk_all = {}
reports = {}

for name, fit_fn in tqdm(candidates, desc="Training Stage1 models"):
    model = fit_fn(X_tr, y_tr)
    score_te = predict_score(model, X_te)

    m = evaluate_metrics(y_te, score_te)
    m["model"] = name
    results.append(m)

    topk = topk_metrics(y_te, score_te)
    topk_all[name] = topk
    topk.to_csv(Path(OUT_DIR_METRICS) / f"{name}_topk.csv", index=False)

    joblib.dump(model, Path(OUT_DIR) / f"{name}.joblib")
    np.save(Path(OUT_DIR_METRICS) / f"{name}_test_scores.npy", score_te)

    thr = np.quantile(score_te, 0.99)
    y_pred = (score_te >= thr).astype(int)

    rep_txt = classification_report(y_te, y_pred, digits=4)
    reports[name] = rep_txt

    print("\n" + "=" * 80)
    print(f"[{name}] threshold=quantile(0.99) -> top 1% as positive")
    print(rep_txt)

results_df = pd.DataFrame(results).sort_values(["prauc", "auc"], ascending=False).reset_index(drop=True)
results_df


Training Stage1 models:   0%|          | 0/3 [00:00<?, ?it/s]


[logit] threshold=quantile(0.99) -> top 1% as positive
              precision    recall  f1-score   support

           0     0.9914    0.9998    0.9956    111855
           1     0.9807    0.5354    0.6927      2088

    accuracy                         0.9913    113943
   macro avg     0.9861    0.7676    0.8441    113943
weighted avg     0.9912    0.9913    0.9900    113943


[hgb] threshold=quantile(0.99) -> top 1% as positive
              precision    recall  f1-score   support

           0     0.9916    1.0000    0.9958    111855
           1     0.9991    0.5455    0.7057      2088

    accuracy                         0.9917    113943
   macro avg     0.9954    0.7727    0.8507    113943
weighted avg     0.9917    0.9917    0.9905    113943

[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_

,auc,prauc,base_rate,model
0,0.991869,0.879642,0.018325,lgb_small
1,0.991171,0.877600,0.018325,hgb
2,0.980794,0.783926,0.018325,logit


---

stage1에서 걸러진 데이터만 2차로 거른다면

In [31]:
import pandas as pd

PASS_PATH = "../STAGE1/artifacts/stage1_pass_ids_test.parquet"
TRAIN2_PATH = "../../5DATA/dataset/TRAIN_STAGE2"
TEST2_PATH  = "../../5DATA/dataset/TEST_STAGE2"

pass_ids = pd.read_parquet(PASS_PATH)["id"].astype("int64")

train2 = pd.read_parquet(TRAIN2_PATH)
test2  = pd.read_parquet(TEST2_PATH)

test2_pass = test2[test2["id"].isin(pass_ids)].copy()

print("train2:", train2.shape)
print("test2:", test2.shape)
print("test2_pass:", test2_pass.shape)
print("pass_rate_in_test2:", len(test2_pass) / len(test2))


train2: (608430, 65)
test2: (113943, 65)
test2_pass: (15276, 65)
pass_rate_in_test2: 0.1340670335167584


In [32]:
OUT_DIR = "artifacts/stage2_models"
OUT_DIR_METRICS = "artifacts/stage2_metrics"

In [33]:
from pathlib import Path 
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUT_DIR_METRICS).mkdir(parents=True, exist_ok=True)

In [34]:
LABEL = "fraud"

def load_stage_df(df, label: str = LABEL, id_col: str = "id"):
    df = df.copy()

    if label not in df.columns:
        raise KeyError(f"Missing label column: {label}")

    drop_cols = [label]
    if id_col in df.columns:
        drop_cols.append(id_col)

    y = df[label].astype(np.int8).to_numpy()
    X = df.drop(columns=drop_cols)

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    return X, y


In [35]:
def topk_metrics(y_true, score, top_pct_list=(0.001, 0.002, 0.005, 0.01)):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(score).astype(float)
    n = len(y)
    base_rate = float(y.mean()) if n else np.nan
    order = np.argsort(-s)
    y_sorted = y[order]

    rows = []
    for p in top_pct_list:
        k = max(int(np.ceil(n * p)), 1)
        top_y = y_sorted[:k]
        prec = float(top_y.mean())
        rec = float(top_y.sum() / max(y.sum(), 1))
        lift = float(prec / base_rate) if base_rate and base_rate > 0 else np.nan
        rows.append({"top_pct": p, "k": k, "precision": prec, "recall": rec, "lift": lift, "base_rate": base_rate})
    return pd.DataFrame(rows)


def evaluate_metrics(y_true, score):
    return {
        "auc": float(roc_auc_score(y_true, score)),
        "prauc": float(average_precision_score(y_true, score)),
        "base_rate": float(np.mean(y_true)),
    }


In [36]:

def fit_logit(X_tr, y_tr):
    model = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced",
        ))
    ])
    model.fit(X_tr, y_tr)
    return model


def fit_hgb(X_tr, y_tr):
    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=200,
        l2_regularization=0.0,
        random_state=42,
    )
    model.fit(X_tr, y_tr)
    return model


def fit_lgb_small(X_tr, y_tr):
    if not HAS_LGB:
        raise RuntimeError("lightgbm is not available in this environment.")
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=6,
        min_data_in_leaf=300,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr)
    return model


def predict_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return 1 / (1 + np.exp(-s))
    return model.predict(X)


In [37]:
X_tr, y_tr = load_stage_df(train2)
X_te, y_te = load_stage_df(test2_pass)

print("train:", X_tr.shape, "test:", X_te.shape)
print("base_rate train:", float(y_tr.mean()), "test:", float(y_te.mean()))


train: (608430, 63) test: (15276, 63)
base_rate train: 0.010794996959387276 test: 0.13177533385703064


In [38]:

from sklearn.metrics import classification_report

candidates = [
    ("logit", fit_logit),
    ("hgb", fit_hgb),
]
if HAS_LGB:
    candidates.append(("lgb_small", fit_lgb_small))

results = []
topk_all = {}
reports = {}

for name, fit_fn in tqdm(candidates, desc="Training Stage1 models"):
    model = fit_fn(X_tr, y_tr)
    score_te = predict_score(model, X_te)

    m = evaluate_metrics(y_te, score_te)
    m["model"] = name
    results.append(m)

    topk = topk_metrics(y_te, score_te)
    topk_all[name] = topk
    topk.to_csv(Path(OUT_DIR_METRICS) / f"{name}_topk.csv", index=False)

    joblib.dump(model, Path(OUT_DIR) / f"{name}.joblib")
    np.save(Path(OUT_DIR_METRICS) / f"{name}_test_scores.npy", score_te)

    thr = np.quantile(score_te, 0.99)
    y_pred = (score_te >= thr).astype(int)

    rep_txt = classification_report(y_te, y_pred, digits=4)
    reports[name] = rep_txt

    print("\n" + "=" * 80)
    print(f"[{name}] threshold=quantile(0.99) -> top 1% as positive")
    print(rep_txt)

results_df = pd.DataFrame(results).sort_values(["prauc", "auc"], ascending=False).reset_index(drop=True)
results_df


Training Stage1 models:   0%|          | 0/3 [00:00<?, ?it/s]


[logit] threshold=quantile(0.99) -> top 1% as positive
              precision    recall  f1-score   support

           0     0.8769    0.9998    0.9343     13263
           1     0.9869    0.0750    0.1394      2013

    accuracy                         0.8780     15276
   macro avg     0.9319    0.5374    0.5369     15276
weighted avg     0.8914    0.8780    0.8296     15276


[hgb] threshold=quantile(0.99) -> top 1% as positive
              precision    recall  f1-score   support

           0     0.8770    1.0000    0.9345     13263
           1     1.0000    0.0760    0.1413      2013

    accuracy                         0.8782     15276
   macro avg     0.9385    0.5380    0.5379     15276
weighted avg     0.8932    0.8782    0.8300     15276

[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_

,auc,prauc,base_rate,model
0,0.967972,0.911418,0.131775,lgb_small
1,0.969140,0.909993,0.131775,hgb
2,0.930576,0.831190,0.131775,logit


In [39]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_recall_curve

def fit_hgb_with_params(X_tr, y_tr, params):
    clf = HistGradientBoostingClassifier(
        loss="log_loss",
        random_state=42,
        **params,
    )
    clf.fit(X_tr, y_tr)
    return clf

def predict_score_hgb(model, X):
    return model.predict_proba(X)[:, 1]

def best_precision_under_min_recall(y_true, score, min_recall=0.60):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    prec, rec, thr = precision_recall_curve(y_true, score)
    thr = np.r_[thr, 1.0]

    ok = rec >= min_recall
    if not np.any(ok):
        return None

    idx = np.argmax(np.where(ok, prec, -1.0))
    return {
        "threshold": float(thr[idx]),
        "precision": float(prec[idx]),
        "recall": float(rec[idx]),
    }

def sample_hgb_params(rng):
    max_depth = rng.choice([3, 5, 7, None])
    max_depth = None if max_depth is None else int(max_depth)

    return {
        "learning_rate": float(rng.choice([0.02, 0.05, 0.1])),
        "max_depth": max_depth,
        "max_leaf_nodes": int(rng.choice([15, 31, 63, 127])),
        "min_samples_leaf": int(rng.choice([20, 50, 100, 200])),
        "l2_regularization": float(rng.choice([0.0, 0.1, 1.0, 5.0, 10.0])),
        "max_bins": int(rng.choice([128, 255])),
    }


def tune_hgb_recall_driven(X_tr, y_tr, X_te, y_te, n_trials=30, min_recall=0.60, seed=42):
    rng = np.random.default_rng(seed)

    rows = []
    best = None

    for t in tqdm(range(n_trials), desc="HGB tuning"):
        params = sample_hgb_params(rng)
        model = fit_hgb_with_params(X_tr, y_tr, params)
        score_te = predict_score_hgb(model, X_te)

        best_row = best_precision_under_min_recall(y_te, score_te, min_recall=min_recall)

        row = {
            "trial": t,
            "ok": best_row is not None,
            "precision": np.nan if best_row is None else best_row["precision"],
            "recall": np.nan if best_row is None else best_row["recall"],
            "threshold": np.nan if best_row is None else best_row["threshold"],
            **params,
        }
        rows.append(row)

        if best_row is not None:
            if (best is None) or (best_row["precision"] > best["precision"]):
                best = {
                    "precision": best_row["precision"],
                    "recall": best_row["recall"],
                    "threshold": best_row["threshold"],
                    "params": params,
                    "model": model,
                }

    trials_df = pd.DataFrame(rows).sort_values(["ok", "precision"], ascending=[False, False]).reset_index(drop=True)
    return best, trials_df

best_hgb, hgb_trials = tune_hgb_recall_driven(
    X_tr, y_tr, X_te, y_te,
    n_trials=40,
    min_recall=0.60,
)

hgb_trials.head(10)

HGB tuning:   0%|          | 0/40 [00:00<?, ?it/s]

,trial,ok,precision,recall,threshold,learning_rate,max_depth,max_leaf_nodes,min_samples_leaf,l2_regularization,max_bins
0,6,True,0.999217,0.633880,0.994589,0.10,NaN,31,100,0.0,255
1,2,True,0.999191,0.613512,0.992492,0.10,7.0,63,200,1.0,128
2,4,True,0.999182,0.606557,0.974004,0.05,NaN,31,200,1.0,128
3,11,True,0.999173,0.600099,0.964899,0.05,NaN,15,50,5.0,128
4,32,True,0.998470,0.648286,0.975790,0.05,7.0,127,20,0.1,128
5,10,True,0.998444,0.637357,0.987395,0.10,NaN,31,200,1.0,128
6,18,True,0.998418,0.626925,0.995682,0.10,5.0,31,200,1.0,128
7,23,True,0.998401,0.620467,0.970105,0.05,5.0,127,20,0.1,128
8,19,True,0.998391,0.616493,0.971679,0.05,7.0,15,20,0.1,128
9,25,True,0.998381,0.612519,0.858181,0.02,NaN,127,50,1.0,255


In [40]:
from sklearn.metrics import classification_report

thr = best_hgb["threshold"]
score_te = predict_score_hgb(best_hgb["model"], X_te)
y_pred = (score_te >= thr).astype(int)

print("best precision under recall constraint")
print("precision:", best_hgb["precision"], "recall:", best_hgb["recall"], "thr:", thr)
print(classification_report(y_te, y_pred, digits=4))


best precision under recall constraint
precision: 0.9992169146436961 recall: 0.6338797814207651 thr: 0.9945889296899292
              precision    recall  f1-score   support

           0     0.9474    0.9999    0.9729     13263
           1     0.9992    0.6339    0.7757      2013

    accuracy                         0.9517     15276
   macro avg     0.9733    0.8169    0.8743     15276
weighted avg     0.9542    0.9517    0.9469     15276



## Stage1 결과

### 임계값 선정 전략

mode: best_precision_under_recall
조건: recall ≥ 0.50

Selected threshold = 0.998300

* top_pct = 0.009425
* precision = 0.9981
* recall = 0.5129

### 분류 리포트

| class | precision | recall | f1-score | support |
| ----- | --------- | ------ | -------- | ------- |
| 0     | 0.9910    | 1.0000 | 0.9955   | 112,113 |
| 1     | 0.9981    | 0.5129 | 0.6776   | 2,096   |

| metric              | value  |
| ------------------- | ------ |
| accuracy            | 0.9910 |
| macro avg precision | 0.9946 |
| macro avg recall    | 0.7564 |
| macro avg f1        | 0.8365 |
| weighted avg f1     | 0.9896 |

### Stage1 → Stage2로 전달된 대상

Stage1은 아래 기준으로 거래를 통과시켰다.

```python
pass_mask = score_te >= thr
pass_ids = id_te[pass_mask]
```

* 전체 거래 수: 114,209
* 통과 거래 수: 약 1,076건
* 통과 비율: 0.9425%

즉, 전체 거래 중 상위 약 0.94%의 고위험 거래만 Stage2로 전달된다.

Stage1의 역할은 다음과 같다.

* 높은 precision 유지
* 최소 50% 이상의 recall 확보
* Stage2에 전달할 거래 수를 대폭 축소
* 대규모 거래를 빠르게 1차 필터링

---

## Stage2 결과

### 임계값 선정 전략

best precision under recall constraint
조건: recall ≥ 0.63

Selected threshold = 0.9945889296899292

* precision = 0.9992169146436961
* recall = 0.6338797814207651

### 분류 리포트

| class | precision | recall | f1-score | support |
| ----- | --------- | ------ | -------- | ------- |
| 0     | 0.9474    | 0.9999 | 0.9729   | 13,263  |
| 1     | 0.9992    | 0.6339 | 0.7757   | 2,013   |

| metric              | value  |
| ------------------- | ------ |
| accuracy            | 0.9517 |
| macro avg precision | 0.9733 |
| macro avg recall    | 0.8169 |
| macro avg f1        | 0.8743 |
| weighted avg f1     | 0.9469 |

---

## 2-Stage 구조 해석

Stage1

* 전체 거래를 대상으로 동작
* 상위 약 1% 고위험 거래만 선별
* precision 0.9981 수준으로 오탐 거의 없음
* recall 약 51% 확보
* 대규모 거래를 경량 모델로 빠르게 필터링

Stage2

* Stage1을 통과한 고위험 거래만 재평가
* recall을 63.39%까지 확장
* precision 0.9992 유지
* 맥락·행동 기반 심층 판단 수행

### 전체 설계 논리

1단계: 고정밀 경량 게이트
2단계: 심층 맥락 기반 정밀 판별

이 구조를 통해

* 오탐 최소화
* 재현율 추가 회복
* 계산 자원 효율화
* 실시간 처리 가능성 확보

라는 목적을 동시에 달성한다.
